In [1]:

import os
# %env NX_CUGRAPH_AUTOCONFIG=True

import networkx as nx
import pandas as pd
from tqdm import tqdm
import json
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

# nx.config.backend_priority = ["cugraph"] # remeber to add this into python script

PATH_TO_CONSTANTS = "../"

#define some constants (file paths)
with open(PATH_TO_CONSTANTS+"constants.json") as f:
    CONSTANTS = json.load(f)
    print(CONSTANTS)



{'graphml': '_graph/graph.graphml', 'nodes_csv': '_graph/kg_nodes.csv', 'oregano_v3': '_graph/OREGANO_V3.tsv', 'cleaned_nodes_csv': '_graph/oregano_clenaed_nodes_new.csv', 'edges_csv': '_graph/kg_edges.csv', 'cleaned_edges_csv': '_graph/kg_edges.csv', 'graphlets': '1-kg_processing/graphlet_extraction/graphlets.json', 'templates': '_generation_templates/templates.jsonl', 'templates_metadata': '_generation_templates/templates_metadata.json', 'generation_templates': '_templates/for_generation.json', 'figures': '_figures/', 'model_paths': '/beegfs/client/default/dl-models/turbomind/', 'llm_outs': '_llm_outs'}


In [2]:
with open(f"{PATH_TO_CONSTANTS}{CONSTANTS['llm_outs']}/FILTER_DATASET/filtered_ds.jsonl", "r") as file:
    ds = [json.loads(line) for line in file]  
    
with open(f"{PATH_TO_CONSTANTS}{CONSTANTS['llm_outs']}/FILTER_DATASET/rejected_ds.jsonl", "r") as file:
    rejected = [json.loads(line) for line in file]  

In [5]:
from collections import defaultdict
ds_by_graphlet = defaultdict( lambda: list())
rejected_by_graphlet = defaultdict( lambda: list())


for item in ds:
    ds_by_graphlet[item['graphlet_id']].append(item)
    
for item in (rejected):
    rejected_by_graphlet[item['graphlet_id']].append(item)

In [6]:
[(k,len(v)+len(rejected_by_graphlet[k])) for k,v in ds_by_graphlet.items()]

[(5, 9218),
 (15, 9223),
 (22, 9233),
 (20, 8927),
 (11, 9268),
 (10, 9237),
 (21, 9046),
 (12, 9175),
 (14, 9056),
 (13, 9270),
 (23, 9001),
 (26, 5006),
 (6, 9161),
 (16, 9114),
 (8, 9314),
 (17, 9244),
 (29, 9141),
 (4, 9358),
 (25, 9160),
 (24, 9046),
 (2, 3461),
 (19, 9260),
 (1, 9336),
 (28, 9191),
 (9, 9235),
 (3, 9229),
 (18, 5659),
 (7, 9228),
 (27, 3475)]

In [22]:
# ds_by_graphlet[27][1]

In [7]:
import random
random.seed(42)

selected_ids = sorted(ds_by_graphlet.keys())


ds_short = []
ds_full =[]
for _id in selected_ids:
    
    valid = random.sample(ds_by_graphlet[_id], 3)
    invalid = random.sample(ds_by_graphlet[_id], 3)
    
    for i in range(len(valid)):
        if i == 0:
            ds_short.append({'id': valid[i]['id'], 'question': valid[i]['question'], 'answer': valid[i]['answer']})
            ds_short.append({'id': invalid[i]['id'], 'question': invalid[i]['question'], 'answer': invalid[i]['answer']})
        else:
            ds_full.append({'id': valid[i]['id'], 'question': valid[i]['question'], 'answer': valid[i]['answer']})
            ds_full.append({'id': invalid[i]['id'], 'question': invalid[i]['question'], 'answer': invalid[i]['answer']})

    
random.shuffle(ds_short)
random.shuffle(ds_full)

In [48]:

for _id in selected_ids:
    print(f"=========================={_id}===================================")
    sample = random.sample(ds_by_graphlet[_id], 1)[0]
    print(sample['question'])
    print(sample['answer'])
    sample = random.sample(rejected_by_graphlet[_id], 1)[0]
    print(sample['question'])
    print(sample['answer'])
    print(sample['filtering_text'])



==========================1===================================
In patients presenting with porokeratosis, what additional ocular complication might they be at risk for, considering shared underlying genetic predispositions?
Patients with porokeratosis may also be at risk for juvenile cataract, as both conditions can be associated with rare genetic disorders. Specifically, Rothmund-Thomson syndrome type 2, which is linked to porokeratosis, also has a known association with the development of juvenile cataract, highlighting the importance of comprehensive ophthalmologic evaluation in individuals with certain rare genetic skin conditions.
In individuals with a predisposition to acute infection-induced encephalopathy, what underlying muscular condition might increase their susceptibility to severe outcomes from common respiratory infections?
Individuals with a susceptibility to acute infection-induced encephalopathy may have an increased risk of severe outcomes from respiratory infections 

In [8]:
import csv 
with open(f"{PATH_TO_CONSTANTS}{CONSTANTS['llm_outs']}/HUMAN_EVAL/ds_short.csv", 'w', newline='') as f: 
    w = csv.DictWriter(f, fieldnames=['id','question','answer'])
    w.writeheader(); w.writerows(ds_short)

with open(f"{PATH_TO_CONSTANTS}{CONSTANTS['llm_outs']}/HUMAN_EVAL/ds_full.csv", 'w', newline='') as f: 
    w = csv.DictWriter(f, fieldnames=['id','question','answer'])
    w.writeheader(); w.writerows(ds_full)

In [9]:

len(ds_short), len(ds_full)

(58, 116)